In [1]:
import os
import sys

ROOT_PATH = '../'
os.chdir(ROOT_PATH)

sys.path.append(os.path.abspath(ROOT_PATH))

lib_path = os.path.abspath(os.path.join(ROOT_PATH, 'lib'))
if lib_path not in sys.path:
  sys.path.append(lib_path)

In [23]:
from types import SimpleNamespace

import numpy as np
import torch

from model.TriheadSegmentationModel import TriheadSegmentationModel

def infuse_args(model: TriheadSegmentationModel):
  current_args = model.yolo_backbone.args if isinstance(model.yolo_backbone.args, dict) else {}

  if 'overlap_mask' not in current_args:
    current_args['overlap_mask'] = True
  current_args['nc'] = 12

  model.yolo_backbone.args = SimpleNamespace(**current_args)

model = TriheadSegmentationModel(
  yolo_pt_path='./weights/yolo26m-sem.pt',
  device='cuda',
  include_depth=False,
  include_normals=False
)
infuse_args(model)

In [24]:
dummy = torch.from_numpy(np.zeros((1, 3, 640, 640), dtype=np.float32)).to('cuda')

dummy.size()

torch.Size([1, 3, 640, 640])

In [25]:
output = model(dummy)[0]

if isinstance(output, tuple):
  for i, o in enumerate(output):
    print(f"output[{i}]: {o.shape}")
else:
  print(f"output: {output.shape}")

output: torch.Size([1, 12, 80, 80])


In [26]:
from peft import get_peft_model, LoraConfig

def create_peft_model(model: TriheadSegmentationModel):
  valid_target_modules = []
  
  backbone_layers = list(model.yolo_backbone.model.children())[:11]
    
  for layer_idx, layer in enumerate(backbone_layers):
    for name, module in layer.named_modules():
      if isinstance(module, torch.nn.Conv2d) and module.groups == 1:
        full_name = f"model.{layer_idx}.{name}" if name else f"model.{layer_idx}"
        valid_target_modules.append(full_name)

  lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    target_modules=valid_target_modules,
    bias="none",
  )
  
  peft_backbone = get_peft_model(model.yolo_backbone, peft_config=lora_config).to('cuda')
  
  for name, param in peft_backbone.named_parameters():
    if 'model.17' in name:
      param.requires_grad = True
  
  peft_backbone.print_trainable_parameters()
  peft_backbone.to('cuda')

  model.yolo_backbone = peft_backbone
  final_model = model.to('cuda')

  return final_model

infuse_args(model)
peft_backbone = create_peft_model(model)

trainable params: 2,137,328 || all params: 14,674,736 || trainable%: 14.5647


In [27]:
for name, param in peft_backbone.named_parameters():
  if 'model.17.classifier' in name or 'model.17.aux_head' in name:
    print(f'{name}: {param.requires_grad}')

yolo_backbone.base_model.model.model.17.classifier.0.conv.weight: True
yolo_backbone.base_model.model.model.17.classifier.0.bn.weight: True
yolo_backbone.base_model.model.model.17.classifier.0.bn.bias: True
yolo_backbone.base_model.model.model.17.classifier.1.weight: True
yolo_backbone.base_model.model.model.17.classifier.1.bias: True
yolo_backbone.base_model.model.model.17.aux_head.0.conv.weight: True
yolo_backbone.base_model.model.model.17.aux_head.0.bn.weight: True
yolo_backbone.base_model.model.model.17.aux_head.0.bn.bias: True
yolo_backbone.base_model.model.model.17.aux_head.1.weight: True
yolo_backbone.base_model.model.model.17.aux_head.1.bias: True
